# 00 — Diagnostika pacientov (izbor >=5 heldout za Monte Carlo)

Presteje za VSAKEGA pacienta: 4 kvadrante (Blood/Tumor x Pre/Post) + st. ekspandiranih
blood-pre klonov (= pozitivni primeri pri eval ekspanzije).

**Kriterij za AUC:** samo dovolj ekspandiranih blood-pre klonov (>=20). Vsi 4 kvadranti
NISO potrebni za AUC (to je le za UMAP graf) — ekspanzijska napoved dela na BloodPre celicah.
Vec ekspandiranih = ozji interval zaupanja AUC.

Ekspanzija (kot eval): klon je EXPANDED ce df_all_tcrs blood-post count > blood-pre count.

## 0. Mount + config

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd, pickle, os
data_parent_folder = '/content/drive/MyDrive/Diploma/data/processed'

with open(os.path.join(data_parent_folder, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
with open(os.path.join(data_parent_folder, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)

col_bloodtumor = data_labels.columns.get_loc('Tissue')
col_prepost    = data_labels.columns.get_loc('Treatment Stage')
col_patient    = data_labels.columns.get_loc('Patient')
col_tcr        = data_labels.columns.get_loc('CDR3(Beta1)')
x_label = data_labels.values
print('celic:', x_label.shape[0], '| pacientov:', len(np.unique(x_label[:, col_patient])))

## 1. Kvadranti + ekspandirani kloni na pacienta

In [ ]:
# ekspanzija kot eval: blood-post count (stolpec 1) > blood-pre count (stolpec 0)
blood_expanded = (df_all_tcrs.iloc[:, 1].fillna(0) > df_all_tcrs.iloc[:, 0].fillna(0)).values

rows = []
for pid in sorted(np.unique(x_label[:, col_patient]).astype(int)):
    m = x_label[:, col_patient] == pid
    q = {}
    for t, tn in [(0,'Blood'),(1,'Tumor')]:
        for s, sn in [(0,'Pre'),(1,'Post')]:
            q[f'{tn}{sn}'] = int((m & (x_label[:,col_bloodtumor]==t) & (x_label[:,col_prepost]==s)).sum())
    all4 = all(q[k] > 0 for k in q)
    # pozitivni primeri pri eval = blood-pre celice tega pacienta ki so EXPANDED
    bb = m & (x_label[:,col_bloodtumor]==0) & (x_label[:,col_prepost]==0)
    exp_bp = int(blood_expanded[x_label[bb, col_tcr].astype(int)].sum()) if bb.sum()>0 else 0
    rows.append({'pid':pid, **q, 'vsi_4_kvadranti':all4,
                 'blood_pre':q['BloodPre'], 'exp_klonov':exp_bp})

df = pd.DataFrame(rows)
df = df.sort_values(['vsi_4_kvadranti','exp_klonov'], ascending=[False,False])
pd.set_option('display.max_rows', 100)
print(df[['pid','BloodPre','BloodPost','TumorPre','TumorPost','vsi_4_kvadranti','exp_klonov']].to_string(index=False))

## 2. Predlog >=5 heldout pacientov

In [ ]:
# === KRITERIJ ZA IZBOR (za ekspanzijski AUC) ===
# POPRAVEK: 'vsi 4 kvadranti' je bil kriterij za UMAP vizualizacijo (2x2 paneli), NE za AUC.
# Ekspanzijska napoved dela SAMO na BloodPre celicah -> primerja z ekspandiranimi kloni.
# Zato je edini pravi kriterij: dovolj BloodPre celic + dovolj ekspandiranih klonov.
# Vec ekspandiranih = vec pozitivnih primerov = ozji interval zaupanja AUC.
MIN_EXP = 20   # minimum ekspandiranih klonov za smiseln recall/AUC

kandidati = df[df['exp_klonov'] >= MIN_EXP].sort_values('exp_klonov', ascending=False)
print(f'Kandidati (>={MIN_EXP} ekspandiranih blood-pre klonov):')
print(kandidati[['pid','BloodPre','TumorPost','exp_klonov','vsi_4_kvadranti']].to_string(index=False))
print()
print('OPOMBA: vsi_4_kvadranti vpliva le na UMAP graf (4 paneli), NE na AUC.')
print('Pacienti z vec ekspandiranih (npr. P23, P26, P18) dajo ZANESLJIVEJSI AUC kot P24 (le 23).')
print()

# Predlog: top N po st. ekspandiranih (najbolj zanesljivi = najvec pozitivnih)
N = 6
top = kandidati['pid'].head(N).tolist()
print(f'PREDLOG heldout (top {N} po st. ekspandiranih): {top}')
print('-> te vpisi v 03/04 notebooke kot heldout_patient = [pid] (en pacient naenkrat)')
print(f'-> Monte Carlo obseg: {len(top)} pacientov x 3 variante x 3 seedi = {len(top)*9} treningov')